# PyTorch FNN Fault Diagnosis without ReLU

이 노트북은 Google Drive의 `data_6_6_6.csv`를 불러와 다음 작업을 수행합니다.

- 입력: a상 전류 6개 + b상 전류 6개 + c상 전류 6개
- 출력: 마지막 열의 Fault ID
- Train/Test 데이터는 동일하게 사용
- 표준화
- PyTorch 기반 FNN 고장진단 모델 학습
- Epoch별 Train/Test loss 저장
- 클래스별 3상 전류 및 Fault ID 그림 저장
- Confusion matrix 저장
- 추정 결과 CSV 저장
- 학습 모델 및 scaler 정보 저장


In [ ]:
# 1. 라이브러리 및 Google Drive 연결

import os
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from google.colab import drive

# GitHub repository를 저장할 위치
REPO_ROOT = "/content/BasicExample_Diagnosis_copy"
RESULT_DIR = "/content/BasicExample_Diagnosis_copy/results"
CSV_PATH = "/content/BasicExample_Diagnosis_copy/data/data_6_6_6.csv" 
# 저장소가 없으면 clone
if not os.path.exists(REPO_ROOT):
    !git clone -q https://github.com/Jaehoon-Shim/BasicExample_Diagnosis.git {REPO_ROOT}
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
# 2. 설정

SEED = 42
EPOCHS = 800
LEARNING_RATE = 1e-3
PRINT_INTERVAL = 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device    :', DEVICE)
print('CSV Path  :', CSV_PATH)
print('Result Dir:', RESULT_DIR)
print('CSV Exists:', os.path.exists(CSV_PATH))


In [ ]:
# 3. CSV 로드 및 전처리

# 헤더가 없는 CSV 파일
df = pd.read_csv(CSV_PATH, header=None)

# 불필요한 인덱스 열 제거
df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')]

# 모든 값을 숫자로 변환하고 결측값 제거
df = df.apply(pd.to_numeric, errors='coerce')
df = df.dropna().reset_index(drop=True)

# 입력 18열 + 마지막 Fault ID 1열 확인
if df.shape[1] != 19:
    raise ValueError(
        'CSV는 입력 18열과 Fault ID 1열로 구성되어야 합니다. '
        f'현재 열 개수: {df.shape[1]}'
    )

feature_columns = (
    [f'Ia_{i + 1}' for i in range(6)]
    + [f'Ib_{i + 1}' for i in range(6)]
    + [f'Ic_{i + 1}' for i in range(6)]
)

df.columns = feature_columns + ['Fault_ID']
df['Fault_ID'] = df['Fault_ID'].astype(int)

class_ids = np.sort(df['Fault_ID'].unique())
NUM_CLASSES = len(class_ids)

if not np.array_equal(class_ids, np.arange(NUM_CLASSES)):
    raise ValueError(
        f'Fault ID는 0부터 연속된 정수여야 합니다. 현재 ID: {class_ids}'
    )

print('Data shape:', df.shape)
print('Fault ID별 데이터 개수:')
print(df['Fault_ID'].value_counts().sort_index())
display(df.head())
display(df.describe())


In [ ]:
# 3-1. 클래스별 3상 전류 플롯

sample_angles = np.array([0, 60, 120, 180, 240, 300])

for fault_id in class_ids:

    # 각 클래스에서 대표 데이터 하나 선택
    sample = (
        df[df['Fault_ID'] == fault_id]
        .iloc[0, :-1]
        .to_numpy(dtype=np.float32)
    )

    # 전체 18열을 3등분
    a_phase_current = sample[0:6]
    b_phase_current = sample[6:12]
    c_phase_current = sample[12:18]

    plt.figure(figsize=(8, 3))

    plt.plot(
        sample_angles,
        a_phase_current,
        marker='o',
        linewidth=2,
        label='a-phase current'
    )

    plt.plot(
        sample_angles,
        b_phase_current,
        marker='o',
        linewidth=2,
        label='b-phase current'
    )

    plt.plot(
        sample_angles,
        c_phase_current,
        marker='o',
        linewidth=2,
        label='c-phase current'
    )

    plt.xlabel('Electrical angle (deg)', fontsize=12)
    plt.ylabel('Phase current', fontsize=12)
    plt.title(f'Fault ID: {fault_id}', fontsize=12)
    plt.xticks(sample_angles, fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(True)
    plt.legend(
        fontsize=12,
        loc='center left',
        bbox_to_anchor=(1.02, 0.5))
    plt.tight_layout()

    original_data_path = os.path.join(
        RESULT_DIR,
        f'fault_id_{fault_id}_phase_current.jpg'
    )

    plt.savefig(original_data_path, dpi=300, bbox_inches='tight')
    plt.show()

    print('Saved:', original_data_path)


In [ ]:
# 4. 입력/출력 구성 및 표준화

X = df.iloc[:, :-1].to_numpy(dtype=np.float32)
y = df.iloc[:, -1].to_numpy(dtype=np.int64)

# 요청에 따라 train/test 동일
X_train = X.copy()
y_train = y.copy()

X_test = X.copy()
y_test = y.copy()

x_scaler = StandardScaler()

X_train_scaled = x_scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = x_scaler.transform(X_test).astype(np.float32)

X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32,
    device=DEVICE
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long,
    device=DEVICE
)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32,
    device=DEVICE
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long,
    device=DEVICE
)

print('X_train:', X_train_tensor.shape)
print('y_train:', y_train_tensor.shape)


In [ ]:
# 5. FNN 모델 정의

class FNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(18, 24),
            nn.Linear(24, 24),
            nn.Linear(24, NUM_CLASSES)
        )

    def forward(self, x):
        return self.model(x)


model = FNNClassifier().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=100
)

print(model)


In [ ]:
# 6. 모델 학습

train_loss_history = []
test_loss_history = []

best_test_loss = float('inf')
best_model_state = None

for epoch in range(1, EPOCHS + 1):

    # Train
    model.train()

    optimizer.zero_grad()

    train_logits = model(X_train_tensor)
    train_loss = criterion(train_logits, y_train_tensor)

    train_loss.backward()
    optimizer.step()

    # Test
    model.eval()

    with torch.no_grad():
        test_logits = model(X_test_tensor)
        test_loss = criterion(test_logits, y_test_tensor)

    train_loss_value = train_loss.item()
    test_loss_value = test_loss.item()

    train_loss_history.append(train_loss_value)
    test_loss_history.append(test_loss_value)

    scheduler.step(test_loss_value)

    if test_loss_value < best_test_loss:
        best_test_loss = test_loss_value
        best_model_state = copy.deepcopy(model.state_dict())

    if epoch == 1 or epoch % PRINT_INTERVAL == 0:
        current_lr = optimizer.param_groups[0]['lr']

        print(
            f'Epoch {epoch:4d}/{EPOCHS} | '
            f'Train Loss: {train_loss_value:.8f} | '
            f'Test Loss: {test_loss_value:.8f} | '
            f'LR: {current_lr:.6f}'
        )

model.load_state_dict(best_model_state)

print()
print('Best Test Loss:', best_test_loss)


In [ ]:
# 7. Fault ID 추정

model.eval()

# Train/Test 추정
with torch.no_grad():
    train_logits = model(X_train_tensor)
    test_logits = model(X_test_tensor)

    y_train_pred = torch.argmax(
        train_logits,
        dim=1
    ).cpu().numpy()

    y_test_pred = torch.argmax(
        test_logits,
        dim=1
    ).cpu().numpy()

    y_test_probability = torch.softmax(
        test_logits,
        dim=1
    ).cpu().numpy()

print('Actual sample   :', y_test[:10])
print('Predicted sample:', y_test_pred[:10])


In [ ]:
# 8. 성능지표

def evaluate(y_true, y_pred, name):
    accuracy = accuracy_score(y_true, y_pred)

    print(f'[{name}]')
    print(f'Accuracy: {accuracy:.8f}')
    print()

    print(
        classification_report(
            y_true,
            y_pred,
            labels=class_ids,
            target_names=[
                f'Fault ID {fault_id}'
                for fault_id in class_ids
            ],
            digits=4,
            zero_division=0
        )
    )

    return accuracy


train_accuracy = evaluate(
    y_train,
    y_train_pred,
    'Train'
)

test_accuracy = evaluate(
    y_test,
    y_test_pred,
    'Test'
)


In [ ]:
# 9. Epoch별 Train/Test loss 저장

epochs = np.arange(1, EPOCHS + 1)

plt.figure(figsize=(5, 5))

plt.plot(epochs, train_loss_history, label='Train Loss')
plt.plot(epochs, test_loss_history, label='Test Loss')

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Cross Entropy Loss', fontsize=12)
plt.title('Train and Test Loss', fontsize=12)
plt.grid(True)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)

plt.tight_layout()

loss_path = os.path.join(
    RESULT_DIR,
    'train_test_loss.jpg'
)

plt.savefig(loss_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', loss_path)


In [ ]:
# 10. 전체 데이터 기준 고장진단 평가 결과

cm = confusion_matrix(
    y_test,
    y_test_pred,
    labels=class_ids
)

plt.figure(figsize=(8, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_ids
)

disp.plot(
    cmap='Blues',
    colorbar=False,
    values_format='d'
)

plt.xlabel('Predicted Fault ID', fontsize=12)
plt.ylabel('Actual Fault ID', fontsize=12)
plt.title('FNN Fault Diagnosis Result', fontsize=12)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()

classification_path = os.path.join(
    RESULT_DIR,
    'fnn_fault_diagnosis_result.jpg'
)

plt.savefig(
    classification_path,
    dpi=300,
    bbox_inches='tight'
)
plt.show()

print('Saved:', classification_path)


In [ ]:
# ============================================================
# 결과 CSV, 학습 이력 CSV, PyTorch 모델(.pt) 저장
# ============================================================

# 추정 결과 CSV 저장
result_df = pd.DataFrame({
    'Actual_Fault_ID': y_test,
    'Predicted_Fault_ID': y_test_pred,
    'Confidence': y_test_probability.max(axis=1)
})

result_csv_path = os.path.join(
    RESULT_DIR,
    'fault_diagnosis_result.csv'
)

result_df.to_csv(result_csv_path, index=False)

# 학습 이력 CSV 저장
history_df = pd.DataFrame({
    'epoch': np.arange(1, EPOCHS + 1),
    'train_loss': train_loss_history,
    'test_loss': test_loss_history
})

history_csv_path = os.path.join(
    RESULT_DIR,
    'train_test_loss.csv'
)

history_df.to_csv(history_csv_path, index=False)

# 모델 가중치 저장
model_path = os.path.join(
    RESULT_DIR,
    'fnn_fault_model.pt'
)

model_cpu = model.to('cpu')
model_cpu.eval()

torch.save(
    model_cpu.state_dict(),
    model_path
)

if os.path.exists(model_path):
    print('모델 저장 성공')
    print('경로:', model_path)
    print('크기:', os.path.getsize(model_path), 'bytes')
else:
    print('모델 저장 실패')

print('x_mean :', x_scaler.mean_.astype(np.float32))
print('x_scale:', x_scaler.scale_.astype(np.float32))


# 11. 학습 모델을 PT로 저장하고 EXE 제작 파일 준비

아래 셀들은 순서대로 실행합니다.

1. 모델 가중치와 입력 표준화 정보를 `fnn_fault_checkpoint.pt` 하나에 저장합니다.
2. 저장한 PT를 다시 로드하여 Colab에서 추론 결과가 같은지 검증합니다.
3. 이후 GUI 또는 임베디드 배포 코드에서 동일한 전처리와 모델을 사용할 수 있습니다.

> Google Colab은 Linux이므로 Windows EXE 자체는 Windows PC에서 빌드해야 합니다.


In [ ]:
# 11-1. 체크포인트 PT 저장 및 다운로드

from google.colab import files

model_cpu = model.to('cpu')
model_cpu.eval()

checkpoint = {
    'model_state_dict': model_cpu.state_dict(),
    'architecture': [18, 64, 128, 128, 64, 32, NUM_CLASSES],
    'x_mean': x_scaler.mean_.astype(np.float32),
    'x_scale': x_scaler.scale_.astype(np.float32),
    'feature_names': feature_columns,
    'class_ids': class_ids.astype(np.int64),
    'input_name': 'a, b, c phase currents',
    'output_name': 'Fault ID'
}

PT_PATH = os.path.join(
    RESULT_DIR,
    'fnn_fault_checkpoint.pt'
)

torch.save(checkpoint, PT_PATH)

print('PT 저장 완료:', PT_PATH)
print('파일 크기:', os.path.getsize(PT_PATH), 'bytes')
files.download(PT_PATH)


In [ ]:
# 11-2. 저장된 PT 재로드 및 추론 검증

class LoadedFNNClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(18, 24), 
            nn.Linear(24, 24), 
            nn.Linear(24, num_classes)
        )

    def forward(self, x):
        return self.model(x)


loaded_checkpoint = torch.load(
    PT_PATH,
    map_location='cpu',
    weights_only=False
)

loaded_model = LoadedFNNClassifier(
    len(loaded_checkpoint['class_ids'])
)

loaded_model.load_state_dict(
    loaded_checkpoint['model_state_dict']
)
loaded_model.eval()


def predict_from_pt(x_values):
    x_values = np.asarray(
        x_values,
        dtype=np.float32
    ).reshape(1, -1)

    if x_values.shape[1] != 18:
        raise ValueError('입력 데이터는 18개여야 합니다.')

    x_scaled = (
        x_values - loaded_checkpoint['x_mean']
    ) / loaded_checkpoint['x_scale']

    x_tensor = torch.tensor(
        x_scaled,
        dtype=torch.float32
    )

    with torch.inference_mode():
        logits = loaded_model(x_tensor)
        probabilities = torch.softmax(logits, dim=1)[0]
        predicted_index = int(probabilities.argmax().item())

    predicted_fault_id = int(
        loaded_checkpoint['class_ids'][predicted_index]
    )

    return predicted_fault_id, probabilities.numpy()


# 첫 번째 데이터로 저장/로드 결과 확인
sample_x = X[0]
sample_actual = int(y[0])
sample_predicted, sample_probability = predict_from_pt(sample_x)

print('PT 로드 성공')
print('실제 Fault ID:', sample_actual)
print('예측 Fault ID:', sample_predicted)
print('클래스별 확률:', np.round(sample_probability, 4))
